# Feature Engineering & Technical Indicators for S&P 500

This notebook transforms the cleaned dataset into a rich, economically meaningful feature set for forecasting, volatility modeling, and regime detection.

In financial time series, **feature quality** often matters more than model complexity. The goal here is to engineer features that capture momentum, volatility dynamics, temporal dependence, seasonality, and market regimes — rather than adding indicators indiscriminately.

## Feature Categories

1. **Lag Features** — Short-term temporal dependencies
2. **Rolling Statistics** — Changing market conditions
3. **Technical Indicators** — Momentum and volatility measures
4. **Calendar Features** — Seasonal and temporal effects
5. **Regime Features** — Volatility states and market regimes

These features will support classical, GARCH, and machine learning models in later notebooks.

## Load Cleaned Dataset

In [ ]:
# Import the needed libraries
import pandas as pd
import numpy as np
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import Markdown, display
import os

pd.options.plotting.backend = 'plotly'
pio.templates.default = 'plotly_dark'
os.makedirs('../reports/figures', exist_ok=True)


In [91]:
# Load cleaned dataset from EDA
df = pd.read_parquet('../data/sp500_cleaned.parquet').copy()

# Preview dataset
df.head()

Price,Open,High,Low,Close,Volume,log_returns
Date,,,,,,
2000-01-04,1455.219971,1455.219971,1397.430054,1399.420044,1009000000,-0.039099
2000-01-05,1399.420044,1413.270020,1377.680054,1402.109985,1085500000,0.001920
2000-01-06,1402.109985,1411.900024,1392.099976,1403.449951,1092300000,0.000955
2000-01-07,1403.449951,1441.469971,1400.729980,1441.469971,1225200000,0.026730
2000-01-10,1441.469971,1464.359985,1441.469971,1457.599976,1064800000,0.011128


In [92]:

# Dataset validation summary function
def dataset_summary(df):
    return {
        'rows': len(df),
        'start': df.index.min().date(),
        'end': df.index.max().date()
    }

summary = dataset_summary(df)

print(f"Rows: {summary['rows']:,}")
print(f"Date range: {summary['start']} → {summary['end']}")

Rows: 6,638
Date range: 2000-01-04 → 2026-05-28


## Loaded Cleaned Dataset

The cleaned dataset produced during EDA is loaded as the foundation for feature engineering.

Using the EDA output ensures:

- Data integrity checks have already been completed  
- A provider-related zero-volume data anomaly has been identified and removed  
- Partial or incomplete latest observations have been excluded (T-1 enforcement)  
- Log returns have been computed and validated  
- Feature engineering begins from a consistent and reproducible baseline  

The dataset is stored in **Parquet** format to preserve structure and data types while improving loading efficiency.

In [93]:
display(Markdown(f"""
## Dataset Validation

The loaded dataset contains:

- **Rows:** {summary['rows']:,}
- **Temporal Coverage:** {summary['start']} → {summary['end']}

This validation confirms that feature engineering is being performed on the expected EDA output and that temporal coverage remains intact before constructing downstream model inputs.
"""))


## Dataset Validation

The loaded dataset contains:

- **Rows:** 6,638
- **Temporal Coverage:** 2000-01-04 → 2026-05-28

This validation confirms that feature engineering is being performed on the expected EDA output and that temporal coverage remains intact before constructing downstream model inputs.


# 1-Lag Features

In [94]:
lags = [1, 2, 3, 5]

for lag in lags:
    df[f'return_lag_{lag}'] = df['log_returns'].shift(lag)
    
# Stationary alternatives to raw price lags
df['close_pct_lag_1'] = df['Close'].pct_change().shift(1)
df['close_pct_lag_2'] = df['Close'].pct_change().shift(2)

# Additional Lagged features
df['range_lag_1'] = (df['High'] - df['Low']).shift(1)
df['volume_lag_1'] = df['Volume'].shift(1)

lag_cols = [c for c in df.columns if '_lag_' in c]
print(f"Created {len(lag_cols)} lag features")

Created 8 lag features


### Lag Features — Short-Term Temporal Memory

Lag features provide models with access to recent market information by introducing delayed versions of returns, price changes, and trading activity.

Following the statistical diagnostics notebook, lag selection was informed by:

- PACF evidence  
- Short-term market dependence  
- Feature usefulness  
- Redundancy reduction  

### Selected Features

**Return Lags**
- `return_lag_1`
- `return_lag_2`
- `return_lag_3`
- `return_lag_5`

**Price Change Lags**
- `close_pct_lag_1`
- `close_pct_lag_2`

**Market Activity Lags**
- `range_lag_1` — Previous trading day intraday range *(High − Low)*  
- `volume_lag_1` — Previous trading day volume  

### Rationale

Lag selection combined statistical diagnostics with financial feature engineering considerations.

PACF analysis of raw returns showed limited directional dependence, largely concentrated in the earliest lags. However, stronger persistence appeared in **squared returns**, where ACF and PACF diagnostics revealed significant short-term volatility dependence.

This indicates an important stylized fact of financial markets:

- Return direction shows weak predictability  
- Volatility and risk exhibit measurable memory  

The selected lag structure therefore balances:

- Short-term information capture  
- Statistical evidence  
- Feature usefulness  
- Reduced redundancy  

Stationary transformations were preferred where appropriate.

Rather than using raw lagged prices, which remain non-stationary and trend-dependent, lagged **percentage price changes** were used to provide recent market-level information in a more stable and scale-independent form.

Additional lagged variables extend beyond returns alone:

- `range_lag_1` captures recent volatility intensity and intraday uncertainty  
- `volume_lag_1` provides liquidity and market participation context  

### Key Insight

Financial markets often show limited predictability in **direction**, but stronger persistence in **risk and volatility**.

Lag features preserve recent market information across multiple dimensions:

- Direction *(returns)*  
- Market movement *(price changes)*  
- Volatility *(range)*  
- Participation *(volume)*  

while avoiding unnecessary lag complexity.

# 2-Rolling Statistics 

In [95]:
windows = [5, 10, 21, 30, 60]

for w in windows:
    # Volatility measures
    df[f'vol_roll_{w}'] = df['log_returns'].rolling(window=w).std()

    # Mean dynamics
    df[f'return_mean_roll_{w}'] = df['log_returns'].rolling(window=w).mean()

    # Momentum (price relative to moving average)
    df[f'momentum_{w}'] = df['Close'] / df['Close'].rolling(window=w).mean() - 1


print(f'Created rolling features across {len(windows)} windows (5 to 60 days)')

Created rolling features across 5 windows (5 to 60 days)


### Rolling Statistics — Multi-Horizon Market Behaviour

Rolling features allow models to observe how market behaviour evolves through time rather than relying on single-period observations.

Multiple rolling windows were constructed using:

- **5 days**
- **10 days**
- **21 days**
- **30 days**
- **60 days**

These windows represent different market horizons commonly used in quantitative finance and risk analysis.

### Window Selection Rationale

**5 & 10 Days**  
Capture short-term behaviour, rapid sentiment shifts, and immediate market reactions.

**21 Days**  
Approximately one trading month, widely used in technical analysis and portfolio monitoring.

**30 Days**  
Common benchmark for rolling volatility and financial risk reporting.

**60 Days**  
Represents a medium-term market horizon, useful for identifying sustained volatility and regime behaviour.

### Features Created

**Rolling Volatility**
- `vol_roll_*`  
Rolling standard deviation of log returns.

**Rolling Mean Returns**
- `return_mean_roll_*`  
Local average return over each window.

**Price-to-Moving-Average Features**
- `momentum_*`  
Measures how far price deviates from its rolling average:

\[
\frac{Close_t}{MA_t} - 1
\]

Positive values indicate price trading above its recent average, while negative values indicate relative weakness.

### Statistical Motivation

These features directly address the findings from Notebook 02.

Statistical diagnostics showed:

- Weak serial dependence in raw returns  
- Strong volatility clustering  
- Persistent conditional heteroskedasticity  

Because market volatility evolves through time, rolling statistics provide models with information about **changing risk conditions** across multiple horizons.

### Key Insight

Markets may exhibit limited predictability in short-term return direction, but volatility and market conditions evolve more gradually.

Rolling features help capture this evolving structure by allowing models to observe:

- Changing volatility  
- Local return behaviour  
- Market strength relative to recent history  
- Multi-horizon regime dynamics

# 3-Technical Indicators

In [96]:
# 1- Relative Strength Index (RSI)
def rsi(series, period=14):
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100 / (1 + rs))
    return rsi

df['rsi_14'] = rsi(df['Close'], 14)

# 2- MACD (Moving Average Convergence Divergence)
ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['macd'] = ema12 - ema26
df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
df['macd_hist'] = df['macd'] - df['macd_signal']

# 3- Bollinger Bands 
df['bb_middle'] = df['Close'].rolling(window=20).mean()
df['bb_std'] = df['Close'].rolling(window=20).std()
df['bb_upper'] = df['bb_middle'] + 2 * df['bb_std']
df['bb_lower'] = df['bb_middle'] - 2 * df['bb_std']
df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle']

# 4- Average True Range (ATR)
high_low = df['High'] - df['Low']
high_close = np.abs(df['High'] - df['Close'].shift())
low_close = np.abs(df['Low'] - df['Close'].shift())
tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df['atr_14'] = tr.rolling(window=14).mean()

print('Technical Indicators created: RSI, MACD, Bollinger Bands, ATR')


Technical Indicators created: RSI, MACD, Bollinger Bands, ATR


### Technical Indicators

Technical indicators were engineered to capture momentum, trend dynamics, and volatility expansion or contraction.

**Indicators Added:**

- **RSI (14)** — Measures momentum by comparing recent gains and losses, often used to identify overbought or oversold conditions.
- **MACD** — Captures the relationship between short- and long-term exponential moving averages, helping represent trend strength and momentum shifts.
- **Bollinger Bands (20, 2)** — Combine rolling mean and volatility to describe price relative to its recent trading range and volatility environment.
- **ATR (14)** — Measures average true range, providing a robust estimate of market range volatility by incorporating intraday movement and price gaps.

These indicators were selected intentionally rather than added indiscriminately. Their inclusion is motivated by the statistical properties identified during diagnostics, particularly **volatility clustering** and **time-varying risk**.

- **RSI** and **MACD** help represent momentum and trend behaviour.
- **Bollinger Bands** and **ATR** capture changing volatility conditions and market uncertainty.

Rather than serving as standalone trading signals, these indicators act as **engineered explanatory features**, allowing downstream models to observe market behaviour through multiple dimensions beyond raw returns alone.

# 4- Calender and Seasonality Features

In [97]:
# Basic Temporal Features
df['day_of_week'] = df.index.dayofweek # 0 = Monday, 6 = Sunday
df['month'] = df.index.month
df['quarter'] = df.index.quarter
df['is_month_end'] = df.index.is_month_end.astype(int)
df['is_quarter_end'] = df.index.is_quarter_end.astype(int)

# Day of the week dummies (one-hot-encoding)
dow_dummies = pd.get_dummies(
    df['day_of_week'],
    prefix='dow',
    drop_first=True,
    dtype=int
)

month_dummies = pd.get_dummies(
    df['month'],
    prefix='month',
    drop_first=True,
    dtype=int
)

df = pd.concat([df, dow_dummies, month_dummies], axis=1)

print("Calendar and seasonality features created")
print(f"Added {len(dow_dummies.columns) + len(month_dummies.columns)} dummy variables")

Calendar and seasonality features created
Added 15 dummy variables


### Calendar & Seasonality Features

Calendar features help models capture systematic temporal patterns, including day-of-week effects and month-of-year seasonality observed during exploratory analysis.

**Features Created:**

**Temporal Features**
- `day_of_week`
- `month`
- `quarter`
- `is_month_end`
- `is_quarter_end`

**One-Hot Encoded Features**
- Day-of-week dummies (`dow_*`)
- Month dummies (`month_*`)

### Rationale

Exploratory analysis identified evidence of recurring seasonal behaviour:

- Tuesday and Wednesday showed relatively stronger average returns.
- September exhibited historically weaker performance.  

These calendar features allow models to learn and adjust for recurring temporal effects that may influence market behaviour.

Day-of-week and month indicators can be particularly relevant because institutional activity often follows predictable timing patterns, including:

- Portfolio rebalancing  
- Reporting cycles  
- Month-end positioning  
- Option expiration effects  
- Seasonal trading behaviour  

### Encoding Choice

Dummy variables (one-hot encoding) are used rather than relying only on raw numeric calendar values.

This prevents models from assuming artificial ordering or distance relationships between categories.

For example:

- Monday is not "halfway" between Sunday and Tuesday  
- September is not numerically "larger" than February in an economic sense  

One-hot encoding allows models to treat each category as a distinct temporal state.

### Key Insight

Calendar features provide **temporal context** rather than price information.

While seasonal effects are rarely strong enough to act as standalone signals, they may improve forecasting and regime identification when combined with momentum, volatility, and market-structure features.

# 5- Regime Features

In [98]:
# Volatility above 1-year historical average:
df['vol_above_avg'] = np.where(
    df['vol_roll_30'].rolling(252).mean().isna(),
    np.nan,
    (
        df['vol_roll_30']
        > df['vol_roll_30'].rolling(252).mean()
    ).astype(int)
)

# Volatility percentile rank (0-1):
df['vol_rank_30'] = (
    df['vol_roll_30'].rolling(window=252).rank(pct=True)
)

# Extreme high volatility regime (top 20%):
df['high_vol_regime'] = np.where(
    df['vol_rank_30'].isna(),
    np.nan,
    (df['vol_rank_30'] > 0.8).astype(int)
)

# Volatility expansion (increasing vs 5 days ago):
df['vol_expansion'] = np.where(
    df['vol_roll_30'].shift(5).isna(),
    np.nan,
    (
        df['vol_roll_30']
        > df['vol_roll_30'].shift(5)
    ).astype(int)
)

# Short-term vs medium-term volatility ratio:
df['vol_ratio_10_60'] = df['vol_roll_10'] / df['vol_roll_60']

print("Volatility regime features created successfully")

Volatility regime features created successfully


### Volatility Regime Features

These features were engineered to help models detect and adapt to changing market risk environments, directly reflecting the strong **volatility clustering** and **conditional heteroskedasticity** identified in Notebook 02.

**Features Created:**

- `vol_above_avg` — Whether current 30-day volatility is above its rolling 1-year historical average.
- `vol_rank_30` — Current volatility expressed as a percentile rank relative to the past year (0–1 scale).
- `high_vol_regime` — Binary flag identifying extreme volatility periods (top 20% historically).
- `vol_expansion` — Detects whether volatility is increasing relative to 5 trading days earlier.
- `vol_ratio_10_60` — Ratio of short-term (10-day) to medium-term (60-day) volatility.

### Why These Features Matter

Financial markets do not exhibit constant risk. Instead, they transition between relatively calm and highly turbulent volatility regimes.

These features provide models with explicit information about:

- Whether market risk is currently elevated.
- Whether volatility is expanding or contracting.
- How current volatility compares with historical conditions.
- Whether short-term stress is building relative to medium-term norms.

Rather than relying solely on raw volatility levels, regime features provide **market-state awareness** and historical context.

This is particularly valuable for:

- **Volatility forecasting** (GARCH-family models)  
- **Risk management and stress detection**  
- **Regime identification**  
- **Anomaly and extreme-event detection** in later project phases.

### Key Insight

Notebook 02 showed that return direction exhibits limited dependence, while volatility displays strong persistence.

These features operationalise that finding by allowing models to learn **how risk evolves through time**, not simply how prices move.

# Missing values handling

In [99]:
# Check for NaN values in my Dataframe:
df.isna().sum().sort_values(ascending=False)

vol_above_avg      280
vol_rank_30        280
high_vol_regime    280
momentum_60         59
vol_ratio_10_60     59
                  ... 
month_6              0
month_5              0
month_12             0
month_10             0
month_11             0
Length: 64, dtype: int64

In [100]:
# Creating the dataframe for modelling
df_model = df.dropna().copy()

In [101]:
# The final shape of the data 
print(f"Rows: {df_model.shape[0]:,}")
print(f"Columns: {df_model.shape[1]}")
df_model.info()

Rows: 6,358
Columns: 64
<class 'pandas.DataFrame'>
DatetimeIndex: 6358 entries, 2001-02-13 to 2026-05-28
Data columns (total 64 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Open                 6358 non-null   float64
 1   High                 6358 non-null   float64
 2   Low                  6358 non-null   float64
 3   Close                6358 non-null   float64
 4   Volume               6358 non-null   int64  
 5   log_returns          6358 non-null   float64
 6   return_lag_1         6358 non-null   float64
 7   return_lag_2         6358 non-null   float64
 8   return_lag_3         6358 non-null   float64
 9   return_lag_5         6358 non-null   float64
 10  close_pct_lag_1      6358 non-null   float64
 11  close_pct_lag_2      6358 non-null   float64
 12  range_lag_1          6358 non-null   float64
 13  volume_lag_1         6358 non-null   float64
 14  vol_roll_5           6358 non-null   float64
 15  return_

In [102]:
# Check for infinite values
np.isinf(df.select_dtypes(include=np.number)).sum().sum()

np.int64(0)

In [103]:
# Save our data into a parquet dataframe to reserve data structure
df_model.to_parquet(
    "../data/sp500_features.parquet"
)

## ✅ Feature Engineering Summary

This notebook transformed the cleaned S&P 500 dataset into a structured feature matrix designed for forecasting, volatility modeling, and market regime analysis.

### Feature Groups Created

**1. Lag Features**  
Recent return history, price changes, trading range, and volume context.

**2. Rolling Statistics**  
Rolling volatility, mean returns, and momentum across multiple horizons (5 to 60 days).

**3. Technical Indicators**  
RSI, MACD, Bollinger Bands, and ATR.

**4. Calendar Features**  
Day-of-week and month effects, including seasonality indicators.

**5. Volatility Regime Features**  
Volatility percentile ranks, high-volatility flags, expansion signals, and short-to-medium-term volatility ratios.

### What These Features Represent

The objective was not to generate as many indicators as possible, but to encode economically meaningful market behaviour into a form that models can evaluate.

These features attempt to capture:

- **Market memory** — How recent history influences future behaviour
- **Momentum and trend** — Whether price is strengthening or weakening relative to recent averages
- **Volatility dynamics** — The persistence and clustering of risk
- **Seasonal patterns** — Calendar-driven effects observed during diagnostics
- **Regime awareness** — Whether the market is currently in a relatively calm or stressed state

These features were deliberately engineered in response to the statistical findings from Notebook 02, particularly the evidence of **fat tails, volatility clustering, and conditional heteroskedasticity**.

### Key Takeaway

The statistical diagnostics showed that directional predictability in returns is limited, while volatility exhibits clear persistence and structure.

The purpose of this notebook was to transform those observations into measurable features that can be tested during the modelling phase.

The next step is to evaluate whether these engineered features improve forecasting accuracy, volatility prediction, and market regime identification.